# CP 长序列容量实验：S=16,384 的同 workload 对照

本节只回答一个问题：在**相同序列长度、batch、dtype 和数据**下，Context Parallel（Ulysses CP）能否降低单 rank 的长序列 activation 峰值，从而避免 OOM。07.04 已说明 CP 的机制：attention 前后通过 AllToAll 在 sequence/head 布局之间交换所有权；它不是数值规约，也不会减少参数、梯度和优化器状态。



## 1. 理论预期（先于 profiling）

设 batch 为 `B`、序列长度为 `S`、hidden size 为 `H`、bf16 为 2 bytes、CP degree 为 `C`。对于一般 GQA 模型，QKV 的 sequence-shaped 大小应按 `B×S_local×(n_q+n_kv+n_kv)×head_dim×bytes` 计算；不能默认写成 `3×B×S×H`。

CP 将每个 rank 的输入 sequence 份额变成 `S_local=S/C`，因此 QKV、Transformer 中的 token-wise activation，以及后续 LM-head/loss 的 token 维都会随 `S_local` 缩小。

CP=2 的 sequence-shaped 部分理论上减半（节省 50%）。参数/梯度/优化器状态、非 sequence-shaped workspace 和 AllToAll buffer 不按此比例减少，所以**完整显存峰值不会必然减半**。AllToAll 把 attention 前的序列份额换成 attention 内的 head 份额，因此 attention 算子内部实际处理的是全局 `S` × `heads/C`；但 attention 输出再换回 local sequence，后续 token-wise 模块和 LM-head/loss 继续使用较短的 sequence。

`peak_no_cp = M_fixed + A_seq`，`peak_cp2 = M_fixed + A_seq/2`，`reduction = (A_seq/2) / peak_no_cp`。

只有 `memory_record.csv` 的双 rank 峰值和 OOM 结果才能给出实际 reduction。

先按模块族拆解 CP 影响范围，再用 Qwen3-1.7B 的参数规模做数量级判断。

### 模块级论证

- 输入 embedding / token hidden state：不直接改 embedding table，但如果上游并行计划保持 sequence shard，后续 hidden state 会沿 `S/C` 路径传播；这类模块不是本次峰值主因，最多是间接受益。
- norm / residual path：同样依赖上游是否保持 sequence shard。它们属于 token-wise 路径，能吃到更短的 local sequence，但不会单独决定峰值。
- QKV projection：这是 CP 的直接收益之一。Qwen3 是 GQA，`n_q=16,n_kv=8,head_dim=128`，所以 QKV 宽度是 `16+8+8=32` 个 head，而不是 `3H`；本 workload 下 no-CP 约 `256 MiB/rank`，CP2 约 `128 MiB/rank`。
- attention core：AllToAll 后变成“全序列 + 半 heads”，所以只随 head 维缩小，典型是 `1/C`，不是 `1/C^2`。`cp2.json` 的 `npu::npu_fusion_attention_v3` 实测输入为 `Q=[32768,8,128]`、`K/V=[32768,4,128]`。
- attention output / post-hook：post-hook 把 attention 输出重新换回 local sequence，后续 token-wise 路径继续看到较短的 sequence。
- token-wise projection / MLP / FFN：CP2 trace 中 `matmul` 输入可见 `[2,8192,2048]` 和 `[2,8192,6144]`；这些模块的 sequence-shaped activation 也随 local sequence 减半。
- LM-head / loss：CP 切分输入后，LM-head 接收到 `[B,S/C,H]`，因此 logits 和 loss/backward buffer 也按 `S/C` 缩小。CP2 的 bf16 logits 约 `4.64 GiB`，loss 中转成 fp32 后约 `9.27 GiB`；no-CP 对应约 `9.27 GiB` 和 `18.55 GiB`。
- AllToAll / allocator buffer：CP 还会引入通信和临时缓冲，这是新增成本，不应算成净省内存。

### Qwen3-1.7B 数量级

- 约 `1.7B` 参数用 bf16 权重、bf16 梯度和 `fp32×2` 的 AdamW 状态约为 `19.0 GiB`；no-CP 的 `dp_shard=2` 约 `9.5 GiB/rank`，CP2 的 `dp_shard=1` 约 `19.0 GiB/rank`。这部分固定状态不受 CP 直接缩减。
- QKV activation 只是几百 MiB，不是整个模型；真正的大块 sequence-shaped activation 是 LM-head 和 loss。CP2 把 bf16 logits 从约 `9.27 GiB` 降至 `4.64 GiB`，把 fp32 loss/backward buffer 从约 `18.55 GiB` 降至 `9.27 GiB`。
- no-CP 的 OOM 请求 `18.55 GiB` 与完整序列 fp32 loss buffer 完全匹配；CP2 的 `9.27 GiB` 峰值分配与 `S/2` 的 loss shape 匹配。
- 因此，CP 的容量收益来自从输入到 LM-head/loss 的多项 sequence-shaped activation 缩小，而不是只靠 QKV 或 attention workspace 的缩减。


## 2. 严格匹配的两卡协议

两次运行必须使用同一 checkpoint、tokenizer、数据顺序/seed、`seq_len=16384`、`local_batch_size=2`、dtype、activation checkpoint 和训练步数；只改变 CP degree 与相应的 data-parallel shard。no-CP 路线使用 `dp_shard=2, cp=1`，global batch size 为 4（示例），每 rank 承担完整 16,384 长度的 sequence；CP2 路线使用 `dp_shard=1, cp=2`，同样取 global batch size 4，每 rank 在 attention 内只处理 8,192 长度的 sequence 份额（attention 内为全局 S × 半 heads）。

这样比较的是同一有效 token workload。容量实验应采双 rank，至少覆盖一个完整 optimizer step。


In [ ]:
import os
original_dir = os.getcwd()
%cd /mnt/workspace/torchtitan-npu
os.environ.update(dict(line.strip().split('=',1) for line in os.popen('source /home/developer/Ascend/cann/set_env.sh && env') if '=' in line))
print(os.getcwd())


## 前置：确认 `sft_qwen3_1_7b_wordle_tnd` 配置已注册

本节的两次训练（no-CP 与 CP2）都使用 `CONFIG=sft_qwen3_1_7b_wordle_tnd`。该 recipe 在 `sft_qwen3_1_7b_wordle` 的基础上用 `_enable_npu_varlen_attention()` 将每层 `inner_attention` 替换为 `NPUVarlenAttention.Config()`，并把 mask 语义设为 `block_causal`。

如果 `torchtitan-npu` 仓库尚未包含此配置，请在 `torchtitan_npu/models/qwen3/config_registry.py` 末尾添加：

```python
def sft_qwen3_1_7b_wordle_tnd() -> TrainerConfig:
    """Wordle SFT with NPUVarlenAttention (TND)."""
    from torchtitan_npu.models.qwen3.tnd_config import _enable_npu_varlen_attention

    base = sft_qwen3_1_7b_wordle()
    assert base.model_spec is not None
    return replace(
        base,
        model_spec=_enable_npu_varlen_attention(base.model_spec),
    )
```



In [ ]:
from torchtitan_npu.models.qwen3.config_registry import sft_qwen3_1_7b_wordle_tnd
print(type(sft_qwen3_1_7b_wordle_tnd()).__name__)


## 3. 采集 no-CP 与 CP2

下面命令仅改变并行拓扑；其余参数保持一致。根据机器显存可先用较少 steps 验证能否启动，再延长到完整 optimizer step。`memory_record.csv`、`communication.json` 和 step trace 必须分别保存，不能覆盖。

In [ ]:
%%bash
set -euo pipefail
for route in no_cp cp2; do
  rm -rf outputs/profile_traces/08_s16384_${route}
done
mkdir -p outputs/profile_traces/08_s16384_no_cp outputs/profile_traces/08_s16384_cp2

# no-CP：两张卡做 data-parallel shard
# S=16384 下该路线预期 OOM；保留日志并继续执行 CP2，而不是让整个教学单元中断。
set +e
HF_HUB_OFFLINE=1 HF_DATASETS_OFFLINE=1 HCCL_IF_BASE_PORT=32100 NGPU=2 \
  MODULE=torchtitan_npu.models.qwen3 CONFIG=sft_qwen3_1_7b_wordle_tnd \
  bash scripts/run_train.sh --training.steps 10 --training.global-batch-size 4 \
  --training.local-batch-size 2 --training.seq_len 16384 \
  --parallelism.data_parallel_replicate_degree 1 --parallelism.data_parallel_shard_degree 2 \
  --parallelism.context_parallel_degree 1 --profiling.enable-profiling \
  --profiling.profile-ranks -1 --profiling.profile-step-start 5 --profiling.profile-step-end 6 \
  --profiling.profile-with-memory --profiling.save-traces-folder profile_traces/08_s16384_no_cp \
  --checkpoint.folder checkpoints/08_s16384_no_cp \
  dataloader:chat-data-loader-config \
  --dataloader.dataset-path parquet \
  --dataloader.data-files assets/data/wordle/train-00000-of-00001.parquet > outputs/profile_traces/08_s16384_no_cp/train.log 2>&1
no_cp_status=$?
set -e
if [[ ${no_cp_status} -eq 0 ]] || ! grep -qiE 'out of memory|OutOfMemoryError' outputs/profile_traces/08_s16384_no_cp/train.log; then
  tail -80 outputs/profile_traces/08_s16384_no_cp/train.log
  echo 'no-CP 运行未按预期以 OOM 结束；请检查 workload 与硬件。' >&2
  exit 1
fi
echo '验证通过：no-CP 在 S=16384 的 backward 中 OOM；开始 CP2。'

# CP2：同样的 workload，sequence 在两张卡间切分
set +e
HF_HUB_OFFLINE=1 HF_DATASETS_OFFLINE=1 HCCL_IF_BASE_PORT=32120 NGPU=2 \
  MODULE=torchtitan_npu.models.qwen3 CONFIG=sft_qwen3_1_7b_wordle_tnd \
  bash scripts/run_train.sh --training.steps 10 --training.global-batch-size 4 \
  --training.local-batch-size 2 --training.seq_len 16384 \
  --parallelism.data_parallel_replicate_degree 1 --parallelism.data_parallel_shard_degree 1 \
  --parallelism.context_parallel_degree 2 --profiling.enable-profiling \
  --profiling.profile-ranks -1 --profiling.profile-step-start 5 --profiling.profile-step-end 6 \
  --profiling.profile-with-memory --profiling.save-traces-folder profile_traces/08_s16384_cp2 \
  --checkpoint.folder checkpoints/08_s16384_cp2 \
  dataloader:chat-data-loader-config \
  --dataloader.dataset-path parquet \
  --dataloader.data-files assets/data/wordle/train-00000-of-00001.parquet > outputs/profile_traces/08_s16384_cp2/train.log 2>&1
cp2_status=$?
set -e
if [[ ${cp2_status} -ne 0 ]]; then
  echo 'CP2 训练异常退出；检查 outputs/profile_traces/08_s16384_cp2/train.log' >&2
fi


In [ ]:
# 理论数量级：分别计算固定状态、单个 sequence-shaped 张量和单次 attention workspace。
# 不把这些数字直接相加，因为它们未必在同一时刻同时驻留。
B, S, H, V = 2, 16384, 2048, 151936
n_q, n_kv, head_dim = 16, 8, 128
bytes_bf16, bytes_fp32 = 2, 4
params = 1.7e9
NPU_CAP = 61.27
model_state = params * (bytes_bf16 + bytes_bf16 + 2 * bytes_fp32) / 2**30

print(f'模型状态总量: {model_state:.1f} GiB (约 1.7B 参数，12 bytes/param)')
print(f'NPU 容量: {NPU_CAP:.2f} GiB\n')

for route, dp_shard, cp in [('no_cp', 2, 1), ('cp2', 1, 2)]:
    s_local = S // cp
    tokens = B * s_local
    fixed = model_state / dp_shard
    qkv = tokens * (n_q + 2 * n_kv) * head_dim * bytes_bf16 / 2**30
    logits = tokens * V * bytes_bf16 / 2**30
    loss_fp32 = tokens * V * bytes_fp32 / 2**30
    print(f'{route}: S_local={s_local}, tokens={tokens}')
    print(f'  固定模型状态:       {fixed:6.2f} GiB/rank')
    print(f'  QKV（GQA，单个路径）: {qkv:6.3f} GiB/rank')
    print(f'  LM-head logits bf16: {logits:6.2f} GiB/rank')
    print(f'  loss/backward fp32:  {loss_fp32:6.2f} GiB/rank')
    print()

print('CP2 attention workspace（operator_memory.csv）: 5.28 GiB/次调用')
print('按 head 数近似外推 no-CP attention workspace: 约 10.56 GiB/次调用')
print('no-CP OOM 日志：44.77 + 18.55 = 63.32 GiB > 61.27 GiB')
print('这些是张量/单次 workspace 数量级，不是可以直接相加的理论峰值。')


## 4. Profiling 验证清单

对 no-CP 和 CP2 的**每个 rank**分别记录：

1. `memory_record.csv` 的 `peak active` 与 `peak reserved`（统一单位），并报告 `max(rank0, rank1)`；
2. 训练日志是否出现 OOM；
3. CP2 的 attention 前/后 AllToAll 次数、payload、device elapsed 与 wait/exposed time；
4. 完整 optimizer step 的 `Stage/Computing/Communication(Not Overlapped)`。

---

# 分析结果

以下单元先介绍 profiling 里会读哪些内容，再用这些证据完成峰值、OOM 与 AllToAll 的判读。

profiling 里主要看四类文件：`memory_record.csv` 负责每个 rank 的 peak active / peak reserved 和 OOM；`operator_memory.csv` 负责定位峰值附近是谁触发了大分配；`trace_view.json` 负责看 attention 输入 shape、AllToAll 调用次数和 device elapsed；如果有 `step_trace_time.csv`，再看 step 级 non-overlapped communication 暴露。

## 5. 读数口径

**显存**：理论 activation 只描述 sequence-shaped 张量（QKV、score）；完整显存以双 rank `memory_record.csv` 的峰值为准。对每个 route 取两个 rank 的最大 peak，避免只看 rank 0 掩盖 rank skew。OOM 以训练日志/退出状态为准。

**AllToAll 消耗**：CP2 的 AllToAll 是 attention 前 Q/K/V 布局交换和 attention 后 output 逆交换，每 step 每 rank 共 448 次调用。从 `trace_view.json` 的 `c10d::alltoall_base_` X 事件读取：调用次数 = 事件条数，device elapsed = `dur` 之和。同一通信在 trace 中出现多个事件（`AllToAll`/`AllToAllBackward` autograd 层、`Enqueue`/`Dequeue`/`HcclAlltoAllV` kernel 层），以 `c10d::alltoall_base_` 为准口径，不重复计数。

**理论对齐**：Qwen3 的 GQA 使 QKV 从约 256 MiB 降至 128 MiB/rank。`trace_view.json` 的 `npu::npu_fusion_attention_v3` Input Dims 给出 head 维减半的实测 shape，说明 attention 内部是“全序列 + 半 heads”，score 只随 head 维缩减 `1/C`，而不是把序列维切成 `1/C`。同时，输入 sequence 的切分也会把 LM-head/loss 的 token 维从 `S` 变成 `S/C`；本次 trace 中 loss fp32 buffer 从 no-CP 的 18.55 GiB 数量级降到 CP2 的 9.27 GiB。

**暴露时间**：AllToAll elapsed 是 collective 级账本；暴露在 optimizer step 上的上界以 `step_trace_time.csv` 的 `Communication(Not Overlapped)` 为准。该值包含同 step 的其他 collective，不等于 AllToAll elapsed 的简单相加（存在 overlap 时会重复计时）。


In [ ]:
from pathlib import Path
import csv, json, os, re
TORCHTITAN_ROOT = Path.cwd()
ROOT = TORCHTITAN_ROOT / 'outputs/profile_traces'
RUNS = {'no_cp': ROOT/'08_s16384_no_cp', 'cp2': ROOT/'08_s16384_cp2'}

def rank_memory_files(root):
    # Ascend 的目录名通常是 hostname_pid；从 CSV 的 Device Type 识别 rank。
    found = {}
    for p in root.rglob('memory_record.csv'):
        with p.open(errors='replace') as f:
            first = next(csv.DictReader(f), None)
        match = re.search(r'NPU:(\d+)', (first or {}).get('Device Type', ''))
        if match is None:
            raise ValueError(f'无法从 {p} 的 Device Type 识别 NPU rank')
        rank = int(match.group(1))
        if rank in found:
            raise ValueError(f'{root}: rank {rank} 有多份 memory_record.csv，先清理或选择一次采集')
        found[rank] = p
    return found

def numeric(v):
    try: return float(str(v).replace(',', '').split()[0])
    except Exception: return None

def peak_memory(path):
    rows = list(csv.DictReader(path.open(errors='replace')))
    vals = {}
    for row in rows:
        for k,v in row.items():
            if v is None: continue
            key = k.lower().replace(' ', '_')
            if any(x in key for x in ('active','reserved','allocated')):
                n = numeric(v)
                if n is not None: vals[key] = max(vals.get(key, 0), n)
    return vals

REPORT = {}
for route, root in RUNS.items():
    ranks = rank_memory_files(root)
    oom_log = root / 'train.log'
    expected_oom = route == 'no_cp' and oom_log.exists() and re.search(r'out of memory|OutOfMemoryError', oom_log.read_text(errors='replace'), re.I)
    if expected_oom and not ranks:
        REPORT[route] = {}
        print('no_cp: OOM 已由 train.log 确认；不存在完整的 memory_record.csv 是预期结果。')
        continue
    if len(ranks) != 2:
        print(f'{route}: 需要 2 份 memory_record.csv，实际 {len(ranks)} 份；跳过峰值对比。')
        REPORT[route] = {}
        continue
    REPORT[route] = {r: peak_memory(path) for r, path in ranks.items()}
    print(route, 'ranks=', sorted(ranks), REPORT[route])


In [ ]:
def route_peak(route):
    per_rank = REPORT.get(route, {})
    keys = set().union(*(x.keys() for x in per_rank.values())) if per_rank else set()
    return {k: max((x.get(k, 0) for x in per_rank.values()), default=0) for k in keys}

peaks = {r: route_peak(r) for r in RUNS}
if not REPORT['no_cp']:
    print('no_cp: S=16384 OOM；容量收益由“不可运行 → CP2 可运行”给出，不能伪造 peak reduction 百分比。')
for metric in sorted(set().union(*(p.keys() for p in peaks.values()))):
    a,b = peaks['no_cp'].get(metric), peaks['cp2'].get(metric)
    if a:
        print(f'{metric}: no_cp={a:.1f}, cp2={b:.1f}, reduction={(a-b)/a:.1%}')
print('注意：单位沿用 CSV；若文件混用 bytes/MB，请先统一单位再计算。')


In [ ]:
# 直接核对 CP2 训练 trace 的 attention 输入 shape：
# Ulysses AllToAll 后每 rank 持 全序列 tokens × heads/C × head_dim —— head 减半即 shape 证据。
import json, re
from pathlib import Path

def load_trace(path):
    raw = path.read_text()
    raw = re.sub(r',(\s*[}\]])', r'\1', raw)  # fix CANN trailing comma
    return json.loads(raw)

traces = sorted(RUNS['cp2'].rglob('trace_view.json'))
assert traces, '未找到 trace_view.json'
trace = load_trace(traces[0])
evs = [e for e in trace
       if e.get('name') == 'npu::npu_fusion_attention_v3' and 'Input Dims' in e.get('args', {})]
print(f'npu::npu_fusion_attention_v3 事件 = {len(evs)} 条')

dims = [x.strip() for x in evs[0]['args']['Input Dims'].split(';\r\n') if x.strip()]
print('Q/K/V/mask dims =', dims[:4])

def mi(tokens, heads, hdim=128, bytes_per_elem=2):
    return tokens * heads * hdim * bytes_per_elem / 2 ** 20

q_d, k_d, _ = dims[:3]
tokens = int(q_d.split(',')[0])
q_h, kv_h = int(q_d.split(',')[1]), int(k_d.split(',')[1])
cp2_mem = mi(tokens, q_h) + 2 * mi(tokens, kv_h)
full_mem = mi(tokens, 16) + 2 * mi(tokens, 8)
print(f'实测（CP2）：Q={q_h} head、K/V={kv_h} head -> QKV 共 {cp2_mem:.0f} MiB')
print(f'全模型（CP=1）：Q=16、K/V=8 head -> QKV 共 {full_mem:.0f} MiB')
print(f'attention 输入 QKV 缩减比例：{1 - cp2_mem/full_mem:.0%}（与 1/C = 50% 一致）')

# 同一 trace 继续核对 LM-head 与 loss 的 token 维：CP2 的 local sequence 应为 S/2。
logit_mm = [e for e in trace if e.get('name') == 'aten::matmul'
            and e.get('args', {}).get('Input Dims', '').endswith('2048,151936')]
loss_ops = [e for e in trace if e.get('name') == 'aten::_log_softmax'
            and e.get('args', {}).get('Input Dims')]
if logit_mm and loss_ops:
    print('LM-head matmul Input Dims =', logit_mm[0]['args']['Input Dims'])
    print('loss _log_softmax Input Dims =', loss_ops[0]['args']['Input Dims'])
    print('CP2 loss token 数 = B×S/2 = 16384；fp32 loss buffer = 9.27 GiB')
    print('no-CP 对应 token 数 = B×S = 32768；fp32 loss buffer = 18.55 GiB')


### 5.1 Attention 路径的内存缩减

CP2 对 attention 路径的 sequence-shaped 张量产生了直接、可量化的缩减。Ulysses 的切分边界要以 AllToAll 为界分两段看（机制见 07.04 与 `npu_varlen_cp.py` 的 pre/post hook）：

- **AllToAll 之前（QKV 投影输出）**：每 rank 持有 `[B, S/C, n_q+n_kv+n_kv, head_dim]`。Qwen3 的 `n_q=16,n_kv=8,head_dim=128`，所以 QKV 理论占用由 256 MiB/rank（CP=1）降至 128 MiB/rank（CP=2），精确减半；
下图是在 Perfetto（打开 `cp2.json`）中选中一个 `npu::npu_fusion_attention_v3` slice 后，右侧 **Details → Arguments → Input Dims** 的实测值：

![CP2 attention 算子输入 shape（Perfetto Details 实测）](./images/08.03_attention_input_dims.png)

图中共 4 行 shape：Q=`32768,8,128`、K=`32768,4,128`、V=`32768,4,128`、mask=`2048,2048`。32768 是两样本 16,384 打包后的全局 token 数（TND 布局），8/4 是 CP2 切分后的 head 数；把 Q 的 8 与实测的 K/V 4 对照全模型 16/8，即可确认 attention 输入在 head 维精确减半。
- **AllToAll 之后（attention 算子输入）**：序列份额换回全局，换来半数的 head。`cp2.json` 中 `npu::npu_fusion_attention_v3` 的 `Input Dims` 实测为 Q=`32768,8,128`、K/V=`32768,4,128`（32768 = 2 条样本 × 16,384 的打包 token 数），即 16 个 Q head → 8、8 个 KV head → 4。按每张量字节 = tokens×heads×128×2B，Q=64 MiB、K/V=32 MiB，QKV 合计 128 MiB，恰为全 head（256 MiB）的一半；
- **Attention score**（若实例化）：形状为 `[B, Nq/C, S, S]`，只随 head 维缩减 `1/C`，CP=2 时减至 1/2，**不是 1/4**——序列维在 attention 内部仍是全局 `S`，Ulysses 没有把 score 的序列维切掉；
- **AllToAll 交换量**：Q/K/V 各一次 pre-attention AllToAll（head 维 ↔ 序列维互换）+ output 一次 post-attention 逆交换；每次 collective 的 payload 等于整张 Q/K/V（= `B·S/C·H·bytes`），交换不减少总字节，只改变每个 rank 持有的形状。

这正是 CP 使长序列从「不可运行」变为「可运行」的核心机制：CP 先把输入切成 `S/C`，使 attention 前的 QKV、Transformer 内的 token-wise activation 以及 attention 后送入 LM-head 的 hidden state 都沿 sequence 维变短；AllToAll 后 attention 内部则处理全局 `S` × `heads/C`。

> **实际峰值一句话**：CP2 的 active peak 为 42,893 MB、reserved peak 为 51,768 MB（两个 rank 一致）。CP2 的 LM-head logits 约 4.64 GiB，进入 loss 的 fp32 buffer 约 9.27 GiB；no-CP 对应约 9.27 GiB 和 18.55 GiB。CP2 虽然把这些 sequence-shaped buffer 减半，但 `dp_shard=1` 的固定模型状态更大，且还有 attention workspace 和其他 live activation，所以整卡峰值不会减半。

In [ ]:
# 峰值归因：区分“触发峰值的单次分配”和“同一时间已经 live 的内存”。
# Allocation Total Active(MB) 是分配完成后的 active 总量，不能把附近每一行的 Size 相加。
for out in sorted(RUNS['cp2'].rglob('ASCEND_PROFILER_OUTPUT/operator_memory.csv')):
    rows = list(csv.DictReader(out.open(errors='replace')))
    with (out.parent / 'memory_record.csv').open(errors='replace') as f:
        dev = next(csv.DictReader(f))['Device Type']
    peak_row = max(rows, key=lambda r: float(r['Allocation Total Active(MB)'] or 0))
    peak_active = float(peak_row['Allocation Total Active(MB)'])
    t_peak = float(peak_row['Allocation Time(us)'])
    trigger = float(peak_row['Size(KB)']) / 1024 / 1024
    before = peak_active / 1024 - trigger
    print(f'{dev}：峰值 active = {peak_active/1024:.2f} GiB')
    print(f'  触发分配 = {peak_row["Name"]} {trigger:.2f} GiB')
    print(f'  该分配前 active 约 {before:.2f} GiB，分配后达到 {peak_active/1024:.2f} GiB')
    near = [r for r in rows if abs(float(r['Allocation Time(us)'] or 0) - t_peak) < 3_000
            and float(r['Size(KB)'] or 0) / 1024 / 1024 >= 8]
    near.sort(key=lambda r: float(r['Allocation Time(us)'] or 0))
    print('  峰值附近的 8 GiB 以上 loss/backward 分配：')
    for r in near:
        print(f'    request={float(r["Size(KB)"])/1024/1024:5.2f} GiB, active_after={float(r["Allocation Total Active(MB)"])/1024:5.2f} GiB, op={r["Name"]}')
    attn = [r for r in rows if r['Name'] == 'npu::npu_fusion_attention_v3'
            and float(r['Size(KB)'] or 0) / 1024 / 1024 > 5]
    print(f'  attention workspace：约 {max(float(r["Size(KB)"])/1024/1024 for r in attn):.2f} GiB/次调用，重复记录 {len(attn)} 条；不能按记录数相加')


## 6. 理论与实测如何对齐

对 `B=2,H=2048,bf16,S=16384`，CP2 把从输入到 LM-head/loss 的 sequence-shaped hidden state 一起压小：AllToAll **前**的 QKV 投影输出由约 256 MiB/rank 降至 128 MiB/rank，LM-head logits 由约 9.27 GiB 降至 4.64 GiB，loss 的 fp32 buffer 由约 18.55 GiB 降至 9.27 GiB；AllToAll **后**的 attention 算子输入则随 head 维减半。`cp2.json` 的 `npu::npu_fusion_attention_v3` 事件给出实测 shape：Q=`[32768, 8, 128]`、K/V=`[32768, 4, 128]`（32768 = 2×16,384 的打包 token 数）——token 维保持全局，head 维从全模型 16/8 降至 8/4，正是 07.04 所述「全局 S × 半 heads」的算子级证据。

完整显存峰值的 reduction 小于 50% 是预期行为：CP2 确实把 LM-head 输入和 loss/backward 的 token 维从 `S` 缩到 `S/2`，但 CP2 的 `dp_shard=1` 使固定模型状态增加约 `9.5 GiB/rank`，并且仍有 attention workspace、FSDP buffer 和其他 live activation。判读时应同时看 attention 输入 shape、LM-head/loss shape 与 `memory_record.csv` 的双 rank 峰值。


## 7. OOM 与 AllToAll 暴露时间

本轮实测使用两张 61.27 GiB NPU、`S=16,384` 与 GBS=4。no-CP 路线在首次 backward 即 OOM：active 已达 44.77 GiB 时仍需再分配 18.55 GiB；该请求正好是完整序列 fp32 loss buffer，因此总需求约为 63.32 GiB，超过 61.27 GiB。CP2 路线两个 rank 的 active peak 均为 42,893.42 MB，reserved peak 为 51,768 MB，且未出现 OOM；慢 rank 的 captured step 中有 448 次 AllToAll（`c10d::alltoall_base_` X 事件），累计 device elapsed 为 43.9 ms，autograd 层 AllToAll/Backward 事件共 672 条。

AllToAll elapsed 来自 `trace_view.json` 的 `c10d::alltoall_base_` X 事件 dur 之和，是 collective 级账本；`step_trace_time.csv` 在本轮 CANN 25.5.5 采集下未生成，非重叠通信暴露上界需在后续开启 step trace 的采集中补充。两次运行必须使用同一 checkpoint、数据顺序与 `seq_len=16384`，rank 编号来自 `memory_record.csv` 的 `Device Type`。


In [ ]:
# AllToAll 账本：从 CP2 训练 trace 的 c10d::alltoall_base_ X 事件统计每 rank 调用次数与 device elapsed。
# 同一通信在 trace 里出现多次（cpu_op/dequeue/enqueue/Hccl kernel），
# 以 c10d::alltoall_base_ 的 X 事件条数作为「调用次数」口径，其 dur 之和为 device elapsed。
import json, re, csv
from pathlib import Path

def device_rank(output):
    with (output / 'memory_record.csv').open(errors='replace') as f:
        first = next(csv.DictReader(f), None) or {}
    match = re.search(r'NPU:(\d+)', first.get('Device Type', ''))
    if match is None:
        raise ValueError(f'无法从 {output} 识别 NPU rank')
    return int(match.group(1))

def load_trace(path):
    raw = path.read_text()
    raw = re.sub(r',(\s*[}\]])', r'\1', raw)
    return json.loads(raw)

def trace_alltoall(path):
    evs = load_trace(path)
    c10d_x = [e for e in evs if e.get('name') == 'c10d::alltoall_base_' and e.get('ph') == 'X']
    autograd = sum(1 for e in evs
                   if e.get('name') in ('AllToAll', 'AllToAllBackward') and e.get('ph') == 'X')
    elapsed_ms = sum(float(e.get('dur', 0)) for e in c10d_x) / 1000
    return len(c10d_x), autograd, elapsed_ms

for trace_path in sorted(RUNS['cp2'].rglob('trace_view.json')):
    rank = device_rank(trace_path.parent)
    c10d, autograd, elapsed = trace_alltoall(trace_path)
    print(f'CP2/rank{rank}: c10d::alltoall_base_ 调用={c10d} 条，'
          f'autograd AllToAll/Backward 事件={autograd} 条，'
          f'device elapsed={elapsed:.3f} ms')

step_rows = sorted(RUNS['cp2'].rglob('step_trace_time.csv'))
if not step_rows:
    print('未找到 step_trace_time.csv：无法给出非重叠通信暴露上界，请确认采集步骤包含 step trace。')
for path in step_rows:
    record = next(csv.DictReader(path.open(errors='replace')))
    print(f'CP2/rank{record["Device_id"]}: stage={float(record["Stage"])/1000:.3f} ms, '
          f'non-overlapped communication={float(record["Communication(Not Overlapped)"])/1000:.3f} ms')
print('AllToAll elapsed 是 collective 账本；暴露在 optimizer step 上的上界以 step trace 的非重叠通信判断。')


## 8. 教学结论：CP 用通信换长上下文容量

CP 的核心价值是**缩减 attention 路径及其前后的 sequence-sharded activation**，使单 rank 能承担更长的序列。在同一 `S=16,384`、GBS=4、Qwen3-1.7B、bf16、两张 61.27 GiB NPU 的 workload 上，本次对照得到以下结论。

**CP2 切分的是从输入到 loss 的一条 sequence-shaped 路径。** 输入切分后，QKV、Transformer 内的 token-wise 模块、LM-head 以及 loss/backward 都接收 `S/2` 的 local sequence；AllToAll 后 attention 算子的输入为实测 Q=`[32768,8,128]`、K/V=`[32768,4,128]`，即全局 token 数不变、head 数减半。

**LM-head/loss 的减半是本次 OOM 转可运行的直接容量证据。** no-CP 的 bf16 logits 约 `9.27 GiB`，loss 中转成 fp32 后需要约 `18.55 GiB`；CP2 的对应大小约为 `4.64 GiB` 和 `9.27 GiB`。这与 no-CP 的失败请求和 CP2 的峰值分配逐项匹配。CP2 整卡峰值仍为 `41.89 GiB active`，因为 `dp_shard=1` 的固定模型状态更大，并且还有 attention workspace、FSDP buffer 和其他 live activation。

**CP 的通信代价已经可以量化，但本轮不能给出非重叠占比。** 每个 CP2 step 每 rank 448 次 AllToAll；慢 rank 的 collective device elapsed 约 44.1 ms。本轮没有生成 `step_trace_time.csv`，因此不能继续声称 wait、非重叠通信或 2.7% step 占比。

**结论边界。** 本实验回答容量问题：CP2 切分从输入到 LM-head/loss 的 sequence-shaped activation，使 `S=16,384` workload 从不可运行变为可运行；同时引入了可观测的 AllToAll 通信。CP 是否更快属于 TPS 问题，需在完整 optimizer-step 协议下用 profiler-off ablation 回答。


## 练习

1. （单选题）本实验的两种路线分别使用哪种并行拓扑？
    A. no-CP：`dp_shard=2, cp=1`；CP2：`dp_shard=1, cp=2`
    B. no-CP：`tensor_parallel=2`；CP2：`context_parallel=2`
    C. no-CP：`dp_shard=1, cp=1`；CP2：`dp_shard=1, cp=1`
    D. no-CP：`dp_replicate=2`；CP2：`dp_shard=2`

2. （判断题）本实验的正确结论应表述为「S=16,384 时 no-CP 不可运行而上 CP2 可运行」，而不是任何「CP 必然更快」的吞吐结论。


In [ ]:
%cd $original_dir


In [ ]:
!cat ./answer/08.03_answer.txt
